1. metareact

In [21]:
import pandas as pd
from rdkit import Chem
def get_reactive_atom_indices(smiles):
    # 解析 SMILES
    mol = Chem.MolFromSmiles(smiles)
    
    # 获取标记为反应位点的原子
    reactive_atoms = []
    for atom in mol.GetAtoms():
        # 检查是否带有反应位点标记
        if atom.HasProp('molAtomMapNumber'):
            reactive_atoms.append(atom.GetIdx())  # 获取原子序号
    
    return reactive_atoms

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles_with_atom_map(smiles: str) -> str:
    try:
        # 将 SMILES 转换为分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("Invalid SMILES string.")
        
        # 提取位点信息（Atom Maps）
        atom_map = {atom.GetIdx(): atom.GetAtomMapNum() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}
        
        # 标准化分子（使用 Standardizer）
        uncharger = rdMolStandardize.Uncharger()  # 去质子化
        mol = uncharger.uncharge(mol)
        
        # 去除多余氢原子
        mol = Chem.RemoveHs(mol)
        
        # 恢复位点信息（Atom Maps）
        for idx, map_num in atom_map.items():
            mol.GetAtomWithIdx(idx).SetAtomMapNum(map_num)
        
        # 返回标准化后的 SMILES
        standardized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return standardized_smiles
    except Exception as e:
        return f"Error: {e}"

def df_col_getatomindices(dfname,col_old,flat=False):
    preds = dfname[col_old].to_list()
    preds = [i.split('|') for i in preds]
    try:
        preds = [list(map(standardize_smiles_with_atom_map,i)) for i in preds]
        ranks = [list(map(get_reactive_atom_indices,i)) for i in preds]
    except:
        ranks = []
        for k in preds:
            preds_part = []
            for x in k:
                try:
                    x = standardize_smiles_with_atom_map(x)
                    x_new = get_reactive_atom_indices(x)
                except:
                    x_new = []
                preds_part.append(x_new)
            ranks.append(preds_part)
    if flat == True:
        ranks_new = []
        for i in ranks:
            list_part = []
            for j in i:
                list_part = list_part + j
            ranks_new.append(list_part)
        return ranks_new
    else:
        return ranks
    
def canonicalsmiles(smi):
    mol = Chem.MolFromSmiles(smi)
    smi_new = Chem.MolToSmiles(mol)
    return smi_new

In [22]:
def assign_ranks(atom_scores):
    # 按分数降序排序，并获取排序后的原子序号
    sorted_atoms = sorted(atom_scores.items(), key=lambda x: x[1], reverse=True)
    
    # 创建一个字典，保存原子序号对应的排名
    rank_dict = {}
    for rank, (atom, _) in enumerate(sorted_atoms):
        rank_dict[atom] = rank + 1
    
    # 保持原子序号从小到大的顺序，并更新为对应的排名
    ranked_atoms = {atom: rank_dict[atom] for atom in sorted(atom_scores)}
    
    return ranked_atoms

In [23]:
df = pd.read_csv('pred_results_metatrans_has_cyp_enzyme.csv')
site_truth = df_col_getatomindices(df,'site_truth',flat=True) 
predict_site = df_col_getatomindices(df,'predict_site') 
df['site_truth_atomidx'] = site_truth
df['predict_site_atomidx'] = predict_site
df['sub_enz'] = df['substrate']+'|' + df['Enzyme']
df

[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Run

[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Running Uncharger
[11:30:30] Run

,Unnamed: 0,substrate,Enzyme,truth,site_truth,predict_site,site_truth_atomidx,predict_site_atomidx,sub_enz
0,0,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP1A2,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,[32],"[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
1,1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP2C9,CC(C)(O)c1cc(O)ccc1CC[C@@H](SCC1(CC(=O)O)CC1)c...,CC(C)(O)c1c[cH:1]ccc1CC[C@@H](SCC1(CC(=O)O)CC1...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,"[6, 32, 10, 40]","[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
2,2,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP2D6,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,[32],"[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
3,3,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP3A4,CC(C)(O)c1cc(O)ccc1CC[C@@H](SCC1(CC(=O)O)CC1)c...,CC(C)(O)c1c[cH:1]ccc1CC[C@@H](SCC1(CC(=O)O)CC1...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,"[6, 32, 10, 40]","[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
4,4,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CYP2D6,CC(C)C=O|CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCC...,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CC[CH2:1]C2)...,CC(C)C[NH:1]Cc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)c...,"[22, 5, 6, 7, 23, 25]","[[4], [1], [25], [4], [23], [5], [22], [19, 23...",CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1|CY...
...,...,...,...,...,...,...,...,...,...
69,69,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,CYP1A2,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1|O=C1CN=...,[18],"[[18], [15], [10, 17, 18], [16], [8], [3, 4], ...",O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1|CYP1A2
70,70,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,CYP3A4,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1|O=C1CN=...,[18],"[[18], [15], [10, 17, 18], [16], [8], [3, 4], ...",O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1|CYP3A4
71,71,O=P1(NCCCl)OCCCN1CCCl,CYP2C9,NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1CCCl,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1([NH:1]CCCl)OC...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1([NH:1]CCCl)OCCC...,"[9, 2]","[[10], [2], [9], [10], [9], [8], [1, 12, 13], ...",O=P1(NCCCl)OCCCN1CCCl|CYP2C9
72,72,O=P1(NCCCl)OCCCN1CCCl,CYP3A4,NP1(=O)OCCCN1CCCl|N[P@@]1(=O)OCCCN1CCCl|O=P1(N...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1(NCCCl)OCC[CH2:1...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1([NH:1]CCCl)OCCC...,"[10, 9, 2]","[[10], [2], [9], [10], [9], [8], [1, 12, 13], ...",O=P1(NCCCl)OCCCN1CCCl|CYP3A4


In [24]:
df.to_pickle('./predict_site_65_has_enzyme.pickle')

2. smartcyp

In [25]:
df_smartcyp = pd.read_pickle('/home/datahouse1/wangyitian/MetaPred/30_main_products/save_code/65/smartcyp_results.pickle')
substrates = df_smartcyp['substrates'].to_list()
substrates = list(map(canonicalsmiles,substrates))
df_smartcyp['substrates'] = substrates
df_smartcyp['sub_enz'] = df_smartcyp['substrates'] + '|CYP' + df_smartcyp['enzyme']
df_smartcyp

,substrates,enzyme,scores,sub_enz
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,3A4,"{0: 17, 1: 22, 2: 24, 3: 23, 4: 20, 5: 3, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...
1,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,2D6,"{0: 18, 1: 22, 2: 24, 3: 23, 4: 20, 5: 1, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...
2,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,2C9,"{0: 18, 1: 22, 2: 24, 3: 23, 4: 20, 5: 1, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...
3,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,3A4,"{0: 23, 1: 28, 2: 23, 3: 25, 4: 30, 5: 19, 6: ...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
4,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,2D6,"{0: 8, 1: 28, 2: 8, 3: 25, 4: 30, 5: 14, 6: 6,...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...
...,...,...,...,...
190,O=P1(NCCCl)OCCCN1CCCl,2D6,"{0: 10, 1: 11, 2: 5, 3: 2, 4: 8, 6: 10, 7: 6, ...",O=P1(NCCCl)OCCCN1CCCl|CYP2D6
191,O=P1(NCCCl)OCCCN1CCCl,2C9,"{0: 10, 1: 11, 2: 5, 3: 2, 4: 8, 6: 10, 7: 6, ...",O=P1(NCCCl)OCCCN1CCCl|CYP2C9
192,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3A4,"{0: 15, 1: 7, 2: 8, 3: 17, 4: 9, 5: 1, 6: 5, 7...",OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1|CYP3A4
193,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,2D6,"{0: 15, 1: 1, 2: 2, 3: 17, 4: 10, 5: 4, 6: 11,...",OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1|CYP2D6


In [26]:
scores_smartcyp = df_smartcyp['scores'].to_list()
smartcyp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in scores_smartcyp]
df_smartcyp['rank'] = smartcyp_sorted_atoms
df_smartcyp

,substrates,enzyme,scores,sub_enz,rank
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,3A4,"{0: 17, 1: 22, 2: 24, 3: 23, 4: 20, 5: 3, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,"[12, 20, 5, 18, 7, 25, 19, 8, 23, 11, 9, 22, 2..."
1,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,2D6,"{0: 18, 1: 22, 2: 24, 3: 23, 4: 20, 5: 1, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,"[5, 18, 12, 20, 7, 25, 14, 26, 19, 8, 11, 22, ..."
2,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,2C9,"{0: 18, 1: 22, 2: 24, 3: 23, 4: 20, 5: 1, 6: 1...",C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,"[5, 12, 18, 20, 7, 25, 14, 19, 8, 11, 22, 23, ..."
3,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,3A4,"{0: 23, 1: 28, 2: 23, 3: 25, 4: 30, 5: 19, 6: ...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[13, 14, 12, 16, 34, 10, 28, 27, 20, 21, 39, 3..."
4,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,2D6,"{0: 8, 1: 28, 2: 8, 3: 25, 4: 30, 5: 14, 6: 6,...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[13, 34, 16, 14, 12, 6, 33, 0, 2, 20, 21, 37, ..."
...,...,...,...,...,...
190,O=P1(NCCCl)OCCCN1CCCl,2D6,"{0: 10, 1: 11, 2: 5, 3: 2, 4: 8, 6: 10, 7: 6, ...",O=P1(NCCCl)OCCCN1CCCl|CYP2D6,"[9, 3, 11, 10, 2, 7, 8, 4, 12, 0, 6, 1]"
191,O=P1(NCCCl)OCCCN1CCCl,2C9,"{0: 10, 1: 11, 2: 5, 3: 2, 4: 8, 6: 10, 7: 6, ...",O=P1(NCCCl)OCCCN1CCCl|CYP2C9,"[9, 3, 11, 10, 2, 7, 8, 4, 12, 0, 6, 1]"
192,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3A4,"{0: 15, 1: 7, 2: 8, 3: 17, 4: 9, 5: 1, 6: 5, 7...",OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1|CYP3A4,"[5, 10, 7, 25, 8, 24, 6, 9, 1, 2, 4, 19, 22, 1..."
193,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,2D6,"{0: 15, 1: 1, 2: 2, 3: 17, 4: 10, 5: 4, 6: 11,...",OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1|CYP2D6,"[1, 2, 14, 5, 10, 19, 22, 13, 15, 7, 25, 8, 24..."


In [27]:
dict_smartcyp_sub2rank = dict(zip(df_smartcyp['sub_enz'].to_list(),smartcyp_sorted_atoms))
dict_smartcyp_sub2rank

{'C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC|CYP3A4': [12,
  20,
  5,
  18,
  7,
  25,
  19,
  8,
  23,
  11,
  9,
  22,
  21,
  10,
  14,
  26,
  0,
  16,
  6,
  17,
  4,
  15,
  1,
  3,
  2,
  13,
  24],
 'C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC|CYP2D6': [5,
  18,
  12,
  20,
  7,
  25,
  14,
  26,
  19,
  8,
  11,
  22,
  23,
  21,
  10,
  9,
  6,
  17,
  0,
  16,
  4,
  15,
  1,
  3,
  2,
  13,
  24],
 'C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC|CYP2C9': [5,
  12,
  18,
  20,
  7,
  25,
  14,
  19,
  8,
  11,
  22,
  23,
  21,
  10,
  9,
  26,
  6,
  17,
  0,
  16,
  4,
  15,
  1,
  3,
  2,
  13,
  24],
 'CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc(/C=C/c2ccc3ccc(Cl)cc3n2)c1|CYP3A4': [13,
  14,
  12,
  16,
  34,
  10,
  28,
  27,
  20,
  21,
  39,
  37,
  11,
  33,
  23,
  25,
  40,
  6,
  7,
  5,
  8,
  31,
  30,
  0,
  2,
  24,
  3,
  18,
  19,
  17,
  35,
  1,
  32,
  4,
  38,
  9,


In [28]:
df_smartcyp.to_pickle('./smartcyp_cyp.pickle')

3. somp

In [29]:
df_somp = pd.read_pickle('/home/datahouse1/wangyitian/MetaPred/30_main_products/save_code/65/somp_results.pickle')
df_somp = df_somp[df_somp['enzyme']!='UGT']
substrates_somp = df_somp['substrates'].to_list()
substrates_somp = list(map(canonicalsmiles,substrates_somp))
df_somp['substrates'] = substrates_somp
df_somp['sub_enz'] = df_somp['substrates'] + '|CYP' + df_somp['enzyme']
scores_somp = df_somp['scores'].to_list()
somp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in scores_somp]
df_somp['rank'] = somp_sorted_atoms
df_somp

,substrates,enzyme,scores,sub_enz,rank
0,O=P1(NCCCl)OCCCN1CCCl,2C9,"{0: 6, 1: 12, 2: 7, 3: 3, 4: 14, 5: 8, 6: 10, ...",O=P1(NCCCl)OCCCN1CCCl|CYP2C9,"[9, 11, 3, 10, 13, 0, 2, 5, 8, 6, 7, 1, 12, 4]"
2,Cn1c(=O)c2c(ncn2C)n(C)c1=O,3A4,"{0: 3, 1: 11, 2: 12, 3: 6, 4: 13, 5: 14, 6: 9,...",Cn1c(=O)c2c(ncn2C)n(C)c1=O|CYP3A4,"[7, 11, 0, 9, 13, 3, 8, 12, 6, 10, 1, 2, 4, 5]"
3,Cc1cccc(C)c1OCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)...,2C9,"{0: 5, 1: 48, 2: 28, 3: 7, 4: 29, 5: 49, 6: 6,...",Cc1cccc(C)c1OCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)...,"[38, 43, 9, 40, 0, 6, 3, 26, 41, 42, 19, 32, 1..."
4,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,1A2,"{0: 45, 1: 27, 2: 55, 3: 41, 4: 13, 5: 21, 6: ...",O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,"[30, 51, 39, 44, 38, 45, 32, 53, 40, 43, 34, 5..."
5,CCOC(=O)c1c(CSc2ccccc2)n(C)c2cc(Br)c(O)c(CN(C)...,3A4,"{0: 8, 1: 4, 2: 29, 3: 22, 4: 27, 5: 28, 6: 26...",CCOC(=O)c1c(CSc2ccccc2)n(C)c2cc(Br)c(O)c(CN(C)...,"[8, 26, 27, 1, 16, 24, 12, 0, 11, 13, 18, 25, ..."
...,...,...,...,...,...
383,COc1c(-c2ccc3cc(NS(C)(=O)=O)ccc3c2)cc(-n2ccc(=...,2D6,"{0: 1, 1: 29, 2: 30, 3: 25, 4: 21, 5: 5, 6: 6,...",COc1c(-c2ccc3cc(NS(C)(=O)=O)ccc3c2)cc(-n2ccc(=...,"[0, 33, 34, 35, 5, 6, 15, 18, 27, 11, 16, 32, ..."
384,CNCC[C@@H](Oc1ccccc1C)c1ccccc1,2C19,"{0: 1, 1: 16, 2: 11, 3: 17, 4: 2, 5: 6, 6: 20,...",CNCC[C@@H](Oc1ccccc1C)c1ccccc1|CYP2C19,"[0, 4, 10, 13, 17, 5, 9, 16, 18, 8, 2, 11, 15,..."
386,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,3A4,"{0: 13, 1: 1, 2: 6, 3: 31, 4: 23, 5: 7, 6: 11,...",C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,"[1, 19, 13, 15, 25, 2, 5, 17, 26, 7, 6, 8, 0, ..."
387,CCCSc1nc(N[C@@H]2C[C@H]2c2ccc(F)c(F)c2)c2nnn([...,2D6,"{0: 4, 1: 5, 2: 28, 3: 2, 4: 33, 5: 34, 6: 35,...",CCCSc1nc(N[C@@H]2C[C@H]2c2ccc(F)c(F)c2)c2nnn([...,"[31, 3, 32, 0, 1, 9, 12, 26, 29, 35, 38, 15, 3..."


In [30]:
df_somp.to_pickle('./somp_cyp.pickle')

In [31]:
dict_somp_sub2rank = dict(zip(df_somp['sub_enz'].to_list(),somp_sorted_atoms))
dict_somp_sub2rank

{'O=P1(NCCCl)OCCCN1CCCl|CYP2C9': [9,
  11,
  3,
  10,
  13,
  0,
  2,
  5,
  8,
  6,
  7,
  1,
  12,
  4],
 'Cn1c(=O)c2c(ncn2C)n(C)c1=O|CYP3A4': [7,
  11,
  0,
  9,
  13,
  3,
  8,
  12,
  6,
  10,
  1,
  2,
  4,
  5],
 'Cc1cccc(C)c1OCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)C[C@H](Cc1ccccc1)NC(=O)[C@H](C(C)C)N1CCCNC1=O|CYP2C9': [38,
  43,
  9,
  40,
  0,
  6,
  3,
  26,
  41,
  42,
  19,
  32,
  14,
  23,
  27,
  39,
  18,
  20,
  31,
  33,
  45,
  44,
  28,
  13,
  11,
  37,
  15,
  2,
  4,
  46,
  47,
  17,
  21,
  30,
  34,
  35,
  49,
  12,
  7,
  22,
  10,
  29,
  36,
  25,
  8,
  16,
  24,
  1,
  5,
  48],
 'O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=O)c3c(OC4OC(CO)C(O)C(O)C4O)cccc31)c1cccc(OC3OC(CO)C(O)C(O)C3O)c1C2=O|CYP1A2': [30,
  51,
  39,
  44,
  38,
  45,
  32,
  53,
  40,
  43,
  34,
  55,
  4,
  18,
  25,
  46,
  36,
  57,
  10,
  11,
  5,
  19,
  22,
  60,
  29,
  50,
  1,
  15,
  24,
  59,
  9,
  13,
  27,
  48,
  7,
  21,
  23,
  61,
  41,
  42,
  3,
  14,
  8,
  12,
  0,
 

4. 整理结果

In [32]:
substrates_total = df['sub_enz'].to_list()
rank_smartcyp = [dict_smartcyp_sub2rank.get(i,'None') for i in substrates_total]
rank_somp= [dict_somp_sub2rank.get(i,'None') for i in substrates_total]
df['rank_smartcyp'] = rank_smartcyp
df['rank_somp'] = rank_somp
df

,Unnamed: 0,substrate,Enzyme,truth,site_truth,predict_site,site_truth_atomidx,predict_site_atomidx,sub_enz,rank_smartcyp,rank_somp
0,0,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP1A2,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,[32],"[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,None,"[14, 35, 29, 6, 7, 28, 34, 25, 24, 10, 12, 5, ..."
1,1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP2C9,CC(C)(O)c1cc(O)ccc1CC[C@@H](SCC1(CC(=O)O)CC1)c...,CC(C)(O)c1c[cH:1]ccc1CC[C@@H](SCC1(CC(=O)O)CC1...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,"[6, 32, 10, 40]","[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[13, 34, 6, 33, 0, 2, 37, 28, 27, 10, 7, 5, 12...","[14, 0, 2, 35, 6, 7, 13, 21, 22, 25, 10, 34, 2..."
2,2,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP2D6,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(C)(O)c1ccccc1CC[C@H](c1cccc(/C=C/c2ccc3ccc(...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,[32],"[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[13, 34, 16, 14, 12, 6, 33, 0, 2, 20, 21, 37, ...","[14, 35, 7, 6, 10, 34, 25, 24, 12, 11, 13, 29,..."
3,3,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CYP3A4,CC(C)(O)c1cc(O)ccc1CC[C@@H](SCC1(CC(=O)O)CC1)c...,CC(C)(O)c1c[cH:1]ccc1CC[C@@H](SCC1(CC(=O)O)CC1...,CC(O)(c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc...,"[6, 32, 10, 40]","[[40], [1], [15, 21], [32, 33, 34, 35, 37, 38]...",CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[13, 14, 12, 16, 34, 10, 28, 27, 20, 21, 39, 3...","[14, 10, 35, 28, 29, 6, 7, 34, 32, 25, 24, 13,..."
4,4,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CYP2D6,CC(C)C=O|CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCC...,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CC[CH2:1]C2)...,CC(C)C[NH:1]Cc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)c...,"[22, 5, 6, 7, 23, 25]","[[4], [1], [25], [4], [23], [5], [22], [19, 23...",CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1|CY...,"[21, 22, 20, 23, 13, 3, 12, 19, 14, 11, 5, 0, ...","[20, 23, 12, 13, 8, 24, 11, 5, 14, 7, 25, 1, 1..."
...,...,...,...,...,...,...,...,...,...,...,...
69,69,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,CYP1A2,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1|O=C1CN=...,[18],"[[18], [15], [10, 17, 18], [16], [8], [3, 4], ...",O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1|CYP1A2,None,"[2, 7, 9, 15, 8, 6, 10, 16, 12, 0, 3, 5, 1, 14..."
70,70,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,CYP3A4,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=N[CH2:1]1|O=C1CN=...,[18],"[[18], [15], [10, 17, 18], [16], [8], [3, 4], ...",O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1|CYP3A4,"[2, 3, 8, 15, 6, 10, 12, 7, 9, 16, 18, 0, 1, 1...","[2, 15, 7, 9, 8, 6, 10, 16, 12, 3, 0, 13, 5, 1..."
71,71,O=P1(NCCCl)OCCCN1CCCl,CYP2C9,NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1CCCl,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1([NH:1]CCCl)OC...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1([NH:1]CCCl)OCCC...,"[9, 2]","[[10], [2], [9], [10], [9], [8], [1, 12, 13], ...",O=P1(NCCCl)OCCCN1CCCl|CYP2C9,"[9, 3, 11, 10, 2, 7, 8, 4, 12, 0, 6, 1]","[9, 11, 3, 10, 13, 0, 2, 5, 8, 6, 7, 1, 12, 4]"
72,72,O=P1(NCCCl)OCCCN1CCCl,CYP3A4,NP1(=O)OCCCN1CCCl|N[P@@]1(=O)OCCCN1CCCl|O=P1(N...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1(NCCCl)OCC[CH2:1...,O=P1(NCCCl)OCCC[N:1]1CCCl|O=P1([NH:1]CCCl)OCCC...,"[10, 9, 2]","[[10], [2], [9], [10], [9], [8], [1, 12, 13], ...",O=P1(NCCCl)OCCCN1CCCl|CYP3A4,"[9, 3, 11, 10, 2, 7, 8, 4, 12, 0, 6, 1]","[9, 11, 3, 7, 8, 13, 5, 12, 4, 0, 6, 2, 10, 1]"


In [33]:
df.to_pickle('./total_cyp.pickle')

In [34]:
def get_unique_numbers(nums, n):
    """
    从列表中提取前 n 个不同的数字。

    参数:
        nums (list): 输入的数字列表。
        n (int): 需要提取的不同数字的数量。

    返回:
        list: 包含前 n 个不同数字的列表。
    """
    unique_nums = []
    seen = set()

    for num in nums:
        if num not in seen:
            unique_nums.append(num)
            seen.add(num)
        if len(unique_nums) == n:
            break

    return unique_nums

In [35]:
df_total = pd.read_pickle('./total_cyp.pickle')

ourmodel_results_old = df_total['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

top 1 accuracy = 0.21621621621621623
top 3 accuracy = 0.4594594594594595
top 5 accuracy = 0.6081081081081081
top 1 accuracy = 0.16216216216216217
top 3 accuracy = 0.4864864864864865
top 5 accuracy = 0.6486486486486487
top 1 accuracy = 0.7027027027027027
top 3 accuracy = 0.8513513513513513
top 5 accuracy = 0.9324324324324325


In [36]:
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.216216,0.162162,0.702703
1,top3_recall,0.459459,0.486486,0.851351
2,top5_recall,0.608108,0.648649,0.932432


5. 按照酶整理结果

In [38]:
df_total['Enzyme'].unique()

array(['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4'], dtype=object)

In [39]:
df_total_cyp1a2 = df_total[df_total['Enzyme'] == 'CYP1A2']

ourmodel_results_old = df_total_cyp1a2['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp1a2['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp1a2['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp1a2['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.0
top 3 accuracy = 0.0
top 5 accuracy = 0.0
top 1 accuracy = 0.14285714285714285
top 3 accuracy = 0.42857142857142855
top 5 accuracy = 0.5714285714285714
top 1 accuracy = 0.7142857142857143
top 3 accuracy = 0.7857142857142857
top 5 accuracy = 0.9285714285714286


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.0,0.142857,0.714286
1,top3_recall,0.0,0.428571,0.785714
2,top5_recall,0.0,0.571429,0.928571


In [41]:
df_total_cyp2c9 = df_total[df_total['Enzyme'] == 'CYP2C9']

ourmodel_results_old = df_total_cyp2c9['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp2c9['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp2c9['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp2c9['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.25
top 3 accuracy = 0.5625
top 5 accuracy = 0.8125
top 1 accuracy = 0.25
top 3 accuracy = 0.5
top 5 accuracy = 0.6875
top 1 accuracy = 0.8125
top 3 accuracy = 0.875
top 5 accuracy = 0.9375


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.2500,0.2500,0.8125
1,top3_recall,0.5625,0.5000,0.8750
2,top5_recall,0.8125,0.6875,0.9375


In [45]:
df_total_cyp2d6 = df_total[df_total['Enzyme'] == 'CYP2D6']

ourmodel_results_old = df_total_cyp2d6['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp2d6['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp2d6['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp2d6['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.29411764705882354
top 3 accuracy = 0.6470588235294118
top 5 accuracy = 0.7647058823529411
top 1 accuracy = 0.0
top 3 accuracy = 0.5294117647058824
top 5 accuracy = 0.7647058823529411
top 1 accuracy = 0.5882352941176471
top 3 accuracy = 0.8823529411764706
top 5 accuracy = 0.9411764705882353


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.294118,0.000000,0.588235
1,top3_recall,0.647059,0.529412,0.882353
2,top5_recall,0.764706,0.764706,0.941176


In [46]:
df_total_cyp3a4 = df_total[df_total['Enzyme'] == 'CYP3A4']

ourmodel_results_old = df_total_cyp3a4['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp3a4['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp3a4['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp3a4['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.25925925925925924
top 3 accuracy = 0.5185185185185185
top 5 accuracy = 0.7037037037037037
top 1 accuracy = 0.2222222222222222
top 3 accuracy = 0.48148148148148145
top 5 accuracy = 0.5925925925925926
top 1 accuracy = 0.7037037037037037
top 3 accuracy = 0.8518518518518519
top 5 accuracy = 0.9259259259259259


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.259259,0.222222,0.703704
1,top3_recall,0.518519,0.481481,0.851852
2,top5_recall,0.703704,0.592593,0.925926
